# AI Character Studio - Colab (free GPU)

Run **ComfyUI + the dashboard together on a free Colab GPU**. You open one public link on your phone or laptop, pick a theme, upload your character photo, and generate. **Nothing is stored on your own computer** except what you tap *download* on; results on the server auto-delete.

### Before you start
1. Menu: **Runtime -> Change runtime type -> Hardware accelerator -> GPU -> Save**.
2. Run each cell top to bottom (click it, press **Shift+Enter**).
3. The last cell prints a **public link** - open it on your phone.

> Free Colab has GPU **time/usage limits** and the session **turns off** when idle or closed (that's what makes it ephemeral). Images are quick; video (SVD) is heavier and slower.
> All generated content is **safe-for-work**.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'NO GPU: go to Runtime -> Change runtime type -> GPU, then re-run.'

## 2. Install ComfyUI + custom nodes
IPAdapter Plus (character reference) and VideoHelperSuite (video output).

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
!pip -q install -r /content/ComfyUI/requirements.txt

cn = '/content/ComfyUI/custom_nodes'
os.makedirs(cn, exist_ok=True)
if not os.path.isdir(f'{cn}/ComfyUI_IPAdapter_plus'):
    !git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git {cn}/ComfyUI_IPAdapter_plus
if not os.path.isdir(f'{cn}/ComfyUI-VideoHelperSuite'):
    !git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git {cn}/ComfyUI-VideoHelperSuite
!pip -q install -r {cn}/ComfyUI-VideoHelperSuite/requirements.txt || true
print('ComfyUI + nodes installed.')

## 3. Download the models
SDXL base, IPAdapter, CLIP-vision, and the SVD image-to-video model. These are a few GB total, so this cell takes a while the first time.

In [ ]:
import os
M = '/content/ComfyUI/models'
for sub in ['checkpoints', 'ipadapter', 'clip_vision']:
    os.makedirs(f'{M}/{sub}', exist_ok=True)

def dl(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 100000:
        print('exists:', dest); return
    !wget -q --show-progress -O "{dest}" "{url}"

# SDXL base checkpoint
dl('https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
   f'{M}/checkpoints/sd_xl_base_1.0.safetensors')

# Stable Video Diffusion (image-to-video)
dl('https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt/resolve/main/svd_xt.safetensors',
   f'{M}/checkpoints/svd_xt.safetensors')

# IPAdapter (SDXL) + CLIP vision
dl('https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors',
   f'{M}/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors')
dl('https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors',
   f'{M}/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors')

print('Model downloads done.')

## 4. Get the AI Character Studio app + install its deps
Set `PROJECT_GIT_URL` to your copy of this project (push the folder to a public GitHub repo). If you leave it blank, the notebook writes a minimal fallback so you can still test, but using your repo is recommended so you get the full app and any tweaks.

In [ ]:
PROJECT_GIT_URL = ''  # e.g. 'https://github.com/yourname/ai-character-studio.git'

import os
os.chdir('/content')
APP = '/content/ai-character-studio'
if PROJECT_GIT_URL:
    if not os.path.isdir(APP):
        !git clone --depth 1 {PROJECT_GIT_URL} {APP}
    else:
        print('App already cloned.')
else:
    print('No PROJECT_GIT_URL set. Please set it to your repo and re-run this cell.')
    print('(The app source lives in your project folder: src/, web/, config/, workflows/.)')

# FFmpeg for reel assembly (present on Colab, but ensure it):
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)

if os.path.isdir(APP):
    !pip -q install -r {APP}/requirements.txt
    print('App deps installed.')

## 5. Start ComfyUI in the background

In [ ]:
import subprocess, time, urllib.request, os
os.chdir('/content/ComfyUI')

comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT,
)

print('Starting ComfyUI (first start loads models, can take a minute)...')
ok = False
for _ in range(90):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3)
        ok = True; break
    except Exception:
        pass
print('ComfyUI is up.' if ok else 'ComfyUI not responding yet - check /content/comfyui.log')

## 6. Launch the dashboard (ephemeral) + get your public link
Runs the web app in **ephemeral mode** (results auto-delete) and opens a public URL via Cloudflare's free quick tunnel. Open the printed link on your phone.

In [ ]:
import os, subprocess, time, re
APP = '/content/ai-character-studio'
assert os.path.isdir(APP), 'Set PROJECT_GIT_URL in cell 4 and run it first.'
os.chdir(APP)

# Ephemeral hosted mode: temp storage, 30 min TTL, talk to local ComfyUI.
env = dict(os.environ,
           AISTUDIO_EPHEMERAL='1',
           AISTUDIO_TTL_SECONDS='1800',
           COMFYUI_HOST='127.0.0.1', COMFYUI_PORT='8188')

# (Re)start the web app, replacing any previous instance from an earlier run.
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
time.sleep(1)
web = subprocess.Popen(['python', '-m', 'uvicorn', 'web.server:app',
                        '--host', '127.0.0.1', '--port', '8000'],
                       env=env, stdout=open('/content/web.log','w'), stderr=subprocess.STDOUT)

# Wait for the web app to actually answer before starting the tunnel.
import urllib.request
web_ok = False
for _ in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/config', timeout=3); web_ok = True; break
    except Exception:
        pass
print('web app up' if web_ok else 'web app slow to start - see /content/web.log')

# Free Cloudflare quick tunnel -> public HTTPS link (no account needed).
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

# Restart tunnel fresh, log to a file, and scan the file (non-blocking + timeout).
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)
open('/content/tunnel.log', 'w').close()
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000',
     '--no-autoupdate', '--logfile', '/content/tunnel.log'],
    stdout=open('/content/tunnel_stdout.log', 'w'), stderr=subprocess.STDOUT)

link = None
pat = re.compile(r'https://[-a-z0-9]+\.trycloudflare\.com')
for _ in range(60):  # up to ~60s
    time.sleep(1)
    for logf in ('/content/tunnel.log', '/content/tunnel_stdout.log'):
        try:
            txt = open(logf, encoding='utf-8', errors='ignore').read()
        except FileNotFoundError:
            continue
        m = pat.search(txt)
        if m:
            link = m.group(0); break
    if link:
        break

print('\n' + '='*56)
if link:
    print('  OPEN THIS ON YOUR PHONE OR LAPTOP:')
    print('  ' + link)
else:
    print('  Link not ready yet. Re-run THIS cell once more.')
    print('  If it still fails, run:  !cat /content/tunnel.log')
print('='*56)
print('Keep this notebook running while you use the app.')

## Done
Open the link above, upload your character photo, pick a theme, tap **Create**, then **Download all**. When you're finished, just stop/close the runtime - everything on the server is discarded.

**Troubleshooting:** if generation errors, check `/content/comfyui.log` (models/nodes) and `/content/web.log` (app).